# 1. Context

This notebook analyzes OCR performance of Tesseract over synthetic generated PDF images across variois degradation levels

# 2. Imports

In [1]:
import pandas as pd
from pathlib import Path
from collections import defaultdict
import sys

In [2]:
import sys

notebook_path = Path()
sys.path.append(str(notebook_path.resolve().parent))

In [3]:
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory

In [4]:
from src.evaluation.metrics import cer, wer
from src.common.constants import language_code_norm_map, language_to_writing_system
from src.common.utils import get_script_results

# 3. Utils

In [5]:
# get csv paths for all language
results_root = Path("../results/tesseract")
results_langs = [x.name.lower() for x in results_root.glob("*") if x.is_dir()]

In [6]:
writing_sys_dict = defaultdict(list)

for language_res in results_langs:
    script = language_to_writing_system.get(language_res)[0]
    writing_sys_dict[script].append(language_res)
script_language_result = pd.Series(writing_sys_dict).to_frame(name='languages')
script_language_result.index.name = 'script'

In [7]:
script_results_avail = script_language_result.index
results_consolidated = []
for script in script_results_avail:
    results_script = get_script_results(script=script, script_lang_df=script_language_result, result_root=results_root)
    results_consolidated.append(results_script)
consolidated_df = pd.concat(results_consolidated)

In [26]:
## Nan output handling
consolidated_df.fillna("", inplace=True)

# 4. Computing WER & CER

## 4.1. UTC Normalisation
For language supprted by `indic-nlp-library` are normalised, where as languages not supported are left as it is


In [8]:
def normalize_text(text: str, lang: str, norm_factory: IndicNormalizerFactory):
    """Normalize text for a given language using indic normalizer"""
    if lang not in language_code_norm_map:
        return text
    code = language_code_norm_map[lang]
    normalizer = norm_factory.get_normalizer(code)
    normalized_text = normalizer.normalize(text)
    return normalized_text

In [20]:
norm_factory = IndicNormalizerFactory()

In [27]:
cols_to_norm = ['ground_truth', 'ocr_output_L_0', 'ocr_output_L_1',
       'ocr_output_L_2', 'ocr_output_L_3']

for col in cols_to_norm:
    consolidated_df[col] = consolidated_df.apply(lambda x: normalize_text(x[col], x['language'], norm_factory), axis=1)

## 4.2. Estimate Error Rates

In [28]:
# computing cer & wer
gt_col = 'ground_truth'
ocr_output_cols = ['ocr_output_L_0', 'ocr_output_L_1', 'ocr_output_L_2', 'ocr_output_L_3']
cols_create_cer = ['cer_l0', 'cer_l1', 'cer_l2', 'cer_l3']
cols_create_wer = ['wer_l0', 'wer_l1', 'wer_l2', 'wer_l3']
for ocr_output_lvl, col_crt_cer in dict(zip(ocr_output_cols, cols_create_cer)).items():
    consolidated_df[col_crt_cer] = consolidated_df[[gt_col, ocr_output_lvl]].apply(lambda x: cer(x[gt_col], x[ocr_output_lvl]), axis=1)
for ocr_output_lvl, col_crt_wer in dict(zip(ocr_output_cols, cols_create_wer)).items():
    consolidated_df[col_crt_wer] = consolidated_df[[gt_col, ocr_output_lvl]].apply(lambda x: wer(x[gt_col], x[ocr_output_lvl]), axis=1)

In [29]:
agg_results = (consolidated_df.groupby(['script', 'language']).agg(CER_AVG_L0=('cer_l0', 'median'),
                                                    CER_AVG_L1=('cer_l1', 'median'),
                                                    CER_AVG_L2=('cer_l2', 'median'),
                                                    CER_AVG_L3=('cer_l3', 'median'),
                                                    WER_AVG_L0=('wer_l0', 'median'),
                                                    WER_AVG_L1=('wer_l1', 'median'),
                                                    WER_AVG_L2=('wer_l2', 'median'),
                                                    WER_AVG_L3=('wer_l3', 'median')
                                                    ).round(3))

In [30]:
agg_results.columns = agg_results.columns.str.upper()

In [31]:
col_ord = ['file_id', 'language', 'script','ground_truth', 'ocr_output_L_0', 'ocr_output_L_1',
       'ocr_output_L_2', 'ocr_output_L_3', 'cer_l0', 'cer_l1', 'cer_l2',
       'cer_l3', 'wer_l0', 'wer_l1', 'wer_l2', 'wer_l3' ]
consolidated_df = consolidated_df[col_ord]

## upper casing column names
consolidated_df.columns = consolidated_df.columns.str.upper()

In [33]:
agg_results.to_clipboard()

In [34]:
consolidated_df.round(3).to_clipboard(index=False)